In [ ]:
import os
import pyreadr
import json
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from notebook_utils import NOTEBOOK_DIR
from cityscaper.utils import read_rds_to_df, resolve_path, sorted_columns, latlon_filter, geojson_rds_to_json

DATA_DIR = NOTEBOOK_DIR.parent / "data"
OUTPUT_DIR = DATA_DIR / "output"

%matplotlib inline

In [ ]:
# Load reference data for exploration
raw_geom_geojson = geojson_rds_to_json(DATA_DIR / "sf_map.RDS")
clean_geom = geojson_to_parcel_bounds(raw_geom_geojson)
geom_series = pd.Series(clean_geom)

fname = DATA_DIR / "five_rezonings_nongeo.RDS"
rezoning_base = read_rds_to_df(DATA_DIR / "five_rezonings_nongeo_unfiltered.RDS", index_cols='mapblklot')
print('\n'.join(sorted_columns(rezoning_base)))
rezoning_base.loc['3561025',['lat', 'lng', 'year', 'pdev_baseline_1yr', 'ex_height2024']]

In [ ]:
geom_select = [-122.43270,37.76874,-122.43060,37.77047]
rezoning_scenario = "F"  # Family rezoning

rezoning_fname = f"rezoning_{rezoning_scenario}_output.RDS"
rezoning_scenario_data = read_rds_to_df(DATA_DIR / rezoning_fname, index_cols='mapblklot')
assert 'height' in rezoning_scenario_data.columns
assert 'pdev' in rezoning_scenario_data.columns

lots_in_region = latlon_filter(rezoning_scenario_data, *geom_select)
development_candidates = lots_in_region[lots_in_region['ZONING'].notnull()].copy()

modeling_variables = [c for c in development_candidates.columns if 'pdev' in c] + \
                     ['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height',
                      'height', 'lot_coverage_discount', 'ground_floor', 'ACRES', 'ex_height2024']
print("\n".join(modeling_variables))
# rezoning_scenario_data.loc['3561025',['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height']]

development_candidates[modeling_variables]

In [ ]:
# Interpreting parcel data from assessor records, exported from R
tax_data = read_rds_to_df("~/src/rezoner/tax_no_geom.rds", index_cols=['parcel_number'])
parcels_of_interest = ["3542061", "3560091", "3560053", "3535043", "3542041", "3535016" ]
cols_of_interest = ["construction_type", "number_of_units", "percent_of_ownership", "property_class_code",
                    "property_class_code_definition", "year_property_built"]
tax_data.loc[parcels_of_interest, cols_of_interest]

In [ ]:
# SB79 assessment of density thresholds
rezoned_sites = rezoning_scenario_data[rezoning_scenario_data['ZONING'].notnull()].copy()
rezoned_sites['du_ac'] = rezoned_sites['expected_units_if_dev'] / rezoned_sites['ACRES']
total_capacity_for_norm = rezoned_sites['expected_units'].sum()
baseline = rezoned_sites.sort_values('du_ac')[['du_ac', 'expected_units']]

threshold = 200
sites_over_threshold = rezoned_sites[rezoned_sites['du_ac'] > threshold]
sites_over_threshold_clipped = sites_over_threshold.copy()
sites_over_threshold_clipped['expected_units_if_dev'] = sites_over_threshold['du_ac'].clip(upper=threshold) * sites_over_threshold['ACRES']
sites_over_threshold_clipped['expected_units'] = sites_over_threshold_clipped['expected_units_if_dev'] * sites_over_threshold_clipped['pdev']
all_sites_clipped = pd.concat([sites_over_threshold_clipped, rezoned_sites[rezoned_sites['du_ac'] <= threshold]])
clipped = all_sites_clipped.sort_values('du_ac')[['du_ac', 'expected_units']]

fig, ax = plt.subplots()
(baseline.groupby('du_ac').sum().cumsum() / total_capacity_for_norm * 100).squeeze().plot(ax=ax, label='full capacity contribution')
(clipped.groupby('du_ac').sum().cumsum() / total_capacity_for_norm * 100).squeeze().plot(ax=ax, label='clipped capacity contribution')
plt.axvline(200, linestyle='--', color='y', label='200 du/ac')
plt.axvline(250, linestyle='--', color='r', label='250 du/ac')
plt.ylabel('Percent of total expected housing units produced, %')
plt.xlabel('Parcel dwelling units per acre (du/ac) at Family Zoning capacity')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/eric/Desktop/sb79_density_thresholds.png')

rezoned_sites['far'] = rezoned_sites['expected_units_if_dev'] * 850 / (rezoned_sites['ACRES'] * 43560)
tmp = rezoned_sites[['height', 'far', 'du_ac']].groupby('height').mean()
tmp['expected_units'] = rezoned_sites[['height', 'expected_units']].groupby('height').sum()
tmp['expected_units'] = rezoned_sites[['height', 'expected_units']].groupby('height').sum()
tmp['portion_of_expected_units'] = tmp['expected_units'] / tmp['expected_units'].sum()
tmp[['far', 'du_ac', 'portion_of_expected_units']].to_csv('~/Desktop/sb79_height_values.csv')
tmp